In [1]:
# Minimal setup
from pathlib import Path
import re
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path exists: {DATA_PATH.exists()}")

Project root: /home/gaurav/python/kaggle/triage
Data path exists: True


In [2]:
# Load CSV and keep only complaint text + acuity target
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.astype(str).str.strip()

TEXT_COL = "chief_complaint_text"
TARGET_CANDIDATES = ["target_triage_acuity", "triage_acuity", "target"]
target_col = next((c for c in TARGET_CANDIDATES if c in df.columns), None)

if target_col is None:
    raise KeyError(
        "Target column not found. "
        f"Tried {TARGET_CANDIDATES}. First columns: {list(df.columns[:15])}"
    )

data = (
    df[[TEXT_COL, target_col]]
    .rename(columns={target_col: "target_triage_acuity"})
    .dropna(subset=[TEXT_COL, "target_triage_acuity"])
    .copy()
)
data["target_triage_acuity"] = data["target_triage_acuity"].astype(int)

display(data.head(10))
print(f"Rows used: {len(data):,}")
display(data["target_triage_acuity"].value_counts().sort_index().rename("count").to_frame())

,chief_complaint_text,target_triage_acuity
0,fever. throat soreness,4
1,"injury, other and unspecified, of foo.... foot...",4
2,fever. cough,4
3,"abdominal pain, cramps, spasms. chest pain. po...",4
4,"injury, other and unspecified, of foo...",4
5,fever,4
6,cough. nasal congestion,3
7,cough,4
8,"injury, other and unspecified of head...",4
9,skin rash,3


Rows used: 58,124


,count
target_triage_acuity,
1,846
2,8597
3,29568
4,16715
5,2398


In [3]:
# Expand common short forms in chief complaint text
ABBR_MAP = {
    "sob": "shortness of breath",
    "cp": "chest pain",
    "abd": "abdominal",
    "ha": "headache",
    "n/v": "nausea and vomiting",
    "s/p": "status post",
    "w/": "with",
    "w/o": "without",
    "fx": "fracture",
    "lac": "laceration",
    "loc": "loss of consciousness",
    "mva": "motor vehicle accident",
    "uti": "urinary tract infection",
    "uri": "upper respiratory infection",
    "usp": "unspecified",
    "foo": "foot",
    "...": "",
    "oth": "other",
    
}

def expand_text(text: str) -> str:
    txt = str(text).lower().strip()
    txt = re.sub(r"\s+", " ", txt)

    # Replace slash abbreviations first.
    for k in ["n/v", "s/p", "w/o", "w/"]:
        txt = re.sub(rf"(?<!\w){re.escape(k)}(?!\w)", ABBR_MAP[k], txt)

    for abbr, full in ABBR_MAP.items():
        if "/" in abbr:
            continue
        txt = re.sub(rf"(?<![a-z0-9]){re.escape(abbr)}(?![a-z0-9])", full, txt)

    txt = re.sub(r"[^a-z0-9\s\-\.,]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

data["text_clean"] = data["chief_complaint_text"].map(expand_text)
display(data[["chief_complaint_text", "text_clean", "target_triage_acuity"]].head(10))

,chief_complaint_text,text_clean,target_triage_acuity
0,fever. throat soreness,fever. throat soreness,4
1,"injury, other and unspecified, of foo.... foot...","injury, other and unspecified, of foot. foot a...",4
2,fever. cough,fever. cough,4
3,"abdominal pain, cramps, spasms. chest pain. po...","abdominal pain, cramps, spasms. chest pain. po...",4
4,"injury, other and unspecified, of foo...","injury, other and unspecified, of foot...",4
5,fever,fever,4
6,cough. nasal congestion,cough. nasal congestion,3
7,cough,cough,4
8,"injury, other and unspecified of head...","injury, other and unspecified of head...",4
9,skin rash,skin rash,3


In [4]:
# DistilBERT (uncased) + CORN ordinal head with threshold calibration
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
from scipy.optimize import minimize
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "nlpie/distil-clinicalbert"
NUM_CLASSES = 5
MAX_LEN = 100
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 0.1
EPOCHS = 2

print(f"Device: {DEVICE}")

class OrdinalTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=96):
        self.texts = texts.astype(str).tolist()
        self.labels = labels.astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["label"] = torch.tensor(self.labels[idx] - 1, dtype=torch.long)
        return item

class OrdinalBert(nn.Module):
    def __init__(self, model_name: str, num_classes: int = 5, dropout: float = 0.5):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.ordinal_head = nn.Linear(hidden, num_classes - 1)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_state = out.last_hidden_state[:, 0]
        return self.ordinal_head(self.dropout(cls_state))

def corn_loss(logits: torch.Tensor, y_idx: torch.Tensor, num_classes: int = 5) -> torch.Tensor:
    # CORN trains each threshold conditionally: P(y > k | y >= k).
    total_loss = torch.tensor(0.0, device=logits.device)
    total_count = 0

    for k in range(num_classes - 1):
        if k == 0:
            mask = torch.ones_like(y_idx, dtype=torch.bool)
        else:
            mask = y_idx >= k

        if mask.sum() == 0:
            continue

        target = (y_idx[mask] > k).float()
        loss_k = F.binary_cross_entropy_with_logits(logits[mask, k], target, reduction="sum")
        total_loss = total_loss + loss_k
        total_count += int(mask.sum().item())

    return total_loss / max(1, total_count)

def corn_cumulative_probs(logits: torch.Tensor) -> torch.Tensor:
    cond_probs = torch.sigmoid(logits)
    return torch.cumprod(cond_probs, dim=1)

def predict_from_logits(logits: torch.Tensor, thresholds=None) -> torch.Tensor:
    cum_probs = corn_cumulative_probs(logits)
    if thresholds is None:
        thresholds = torch.full((NUM_CLASSES - 1,), 0.5, device=cum_probs.device, dtype=cum_probs.dtype)
    else:
        thresholds = torch.as_tensor(thresholds, device=cum_probs.device, dtype=cum_probs.dtype)
    return (cum_probs > thresholds).sum(dim=1).long() + 1

def find_optimal_thresholds(logits: torch.Tensor, y_true):
    probs = corn_cumulative_probs(logits).detach().cpu().numpy()
    y_true = np.asarray(y_true)

    def objective(candidate):
        candidate = np.sort(np.clip(candidate, 0.01, 0.99))
        preds = (probs > candidate.reshape(1, -1)).sum(axis=1) + 1
        return -cohen_kappa_score(y_true, preds, weights="quadratic")

    result = minimize(
        objective,
        x0=np.full(NUM_CLASSES - 1, 0.5),
        method="Nelder-Mead",
        options={"maxiter": 200, "xatol": 1e-3, "fatol": 1e-4},
    )
    thresholds = np.sort(np.clip(result.x, 0.01, 0.99))
    return thresholds, result

def run_epoch(
    model,
    loader,
    optimizer=None,
    scheduler=None,
    desc="epoch",
    thresholds=None,
    return_logits=False,
 ):
    is_train = optimizer is not None
    model.train(is_train)

    all_true, all_pred, all_logits = [], [], []
    total_loss = 0.0

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        progress = tqdm(loader, desc=desc, leave=False)
        for batch in progress:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            y_idx = batch["label"].to(DEVICE)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = corn_loss(logits, y_idx, num_classes=NUM_CLASSES)

            if is_train:
                loss.backward()
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()

            pred_idx = predict_from_logits(logits.detach(), thresholds=thresholds).cpu().numpy()
            true_idx = (y_idx.detach().cpu().numpy() + 1)
            all_pred.extend(pred_idx.tolist())
            all_true.extend(true_idx.tolist())
            total_loss += loss.item() * input_ids.size(0)

            if return_logits:
                all_logits.append(logits.detach().cpu())

            avg_loss_so_far = total_loss / max(1, len(all_true))
            progress.set_postfix(loss=f"{avg_loss_so_far:.4f}")

    epoch_loss = total_loss / len(loader.dataset)
    qwk = cohen_kappa_score(all_true, all_pred, weights="quadratic")
    all_logits_tensor = torch.cat(all_logits, dim=0) if all_logits else torch.empty((0, NUM_CLASSES - 1))
    return epoch_loss, qwk, all_true, all_pred, all_logits_tensor

Device: cuda


In [5]:
X = data["text_clean"].fillna("")
y = data["target_triage_acuity"].astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
 )

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_full,
 )

print(f"Train size: {len(X_train):,} | Val size: {len(X_val):,} | Test size: {len(X_test):,}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = OrdinalTextDataset(X_train, y_train, tokenizer, max_len=MAX_LEN)
val_ds = OrdinalTextDataset(X_val, y_val, tokenizer, max_len=MAX_LEN)
test_ds = OrdinalTextDataset(X_test, y_test, tokenizer, max_len=MAX_LEN)

class_counts = y_train.value_counts().sort_index()
class_count_tensor = torch.tensor(
    [class_counts.get(i, 1) for i in range(1, NUM_CLASSES + 1)],
    dtype=torch.float32,
 )
class_weights = 1.0 / torch.sqrt(class_count_tensor)
sample_weights = class_weights[(y_train.to_numpy(dtype=int) - 1)]
train_sampler = WeightedRandomSampler(
    weights=sample_weights.detach().cpu().double().tolist(),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

model = OrdinalBert(MODEL_NAME, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = max(1, len(train_loader) * EPOCHS)
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print("Trainable modules: full model")

best_val_qwk = -1.0
best_state = copy.deepcopy(model.state_dict())
best_thresholds = np.full(NUM_CLASSES - 1, 0.5)

for epoch in tqdm(range(1, EPOCHS + 1), desc="Epochs", leave=False):
    tr_loss, tr_qwk, _, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        desc=f"Train e{epoch}",
    )
    val_loss, val_qwk_default, val_true, val_pred_default, val_logits = run_epoch(
        model,
        val_loader,
        desc=f"Val e{epoch}",
        return_logits=True,
    )
    epoch_thresholds, _ = find_optimal_thresholds(val_logits, val_true)
    val_pred_tuned = predict_from_logits(val_logits, thresholds=epoch_thresholds).cpu().numpy()
    val_qwk_tuned = cohen_kappa_score(val_true, val_pred_tuned, weights="quadratic")

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={tr_loss:.4f} train_qwk={tr_qwk:.4f} | "
        f"val_loss={val_loss:.4f} val_qwk@0.5={val_qwk_default:.4f} val_qwk@tuned={val_qwk_tuned:.4f}"
    )
    print(f"  tuned thresholds: {np.round(epoch_thresholds, 3).tolist()}")

    if val_qwk_tuned > best_val_qwk:
        best_val_qwk = val_qwk_tuned
        best_state = copy.deepcopy(model.state_dict())
        best_thresholds = epoch_thresholds.copy()

model.load_state_dict(best_state)

print("\nFinal summary (best validation epoch)")
print(f"Best validation QWK: {best_val_qwk:.4f}")
print(f"Calibrated thresholds: {np.round(best_thresholds, 4).tolist()}")

test_loss, test_qwk_default, test_true, test_pred_default, test_logits = run_epoch(
    model,
    test_loader,
    desc="Test @ 0.5",
    return_logits=True,
 )
test_pred_tuned = predict_from_logits(test_logits, thresholds=best_thresholds).cpu().numpy()
test_qwk_tuned = cohen_kappa_score(test_true, test_pred_tuned, weights="quadratic")

print(f"Test loss: {test_loss:.4f}")
print(f"Test QWK @ 0.5: {test_qwk_default:.4f}")
print(f"Test QWK @ tuned thresholds: {test_qwk_tuned:.4f}")

print("\nClassification report (test, tuned thresholds):")
print(classification_report(test_true, test_pred_tuned, digits=4))

labels = [1, 2, 3, 4, 5]
cm = confusion_matrix(test_true, test_pred_tuned, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{i}" for i in labels], columns=[f"pred_{i}" for i in labels])
display(cm_df)

Train size: 37,199 | Val size: 9,300 | Test size: 11,625


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpie/distil-clinicalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstr

Trainable modules: full model


Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

...

Val e2:   0%|          | 0/582 [00:00<?, ?it/s]

Epoch 2/2 | train_loss=0.3882 train_qwk=0.4970 | val_loss=0.3361 val_qwk@0.5=0.4570 val_qwk@tuned=0.4589
  tuned thresholds: [0.494, 0.495, 0.496, 0.505]

Final summary (best validation epoch)
Best validation QWK: 0.4589
Calibrated thresholds: [0.4938, 0.4951, 0.4961, 0.5047]


Test @ 0.5:   0%|          | 0/727 [00:00<?, ?it/s]

Test loss: 0.3382
Test QWK @ 0.5: 0.4515
Test QWK @ tuned thresholds: 0.4529

Classification report (test, tuned thresholds):
              precision    recall  f1-score   support

           1     0.3059    0.1538    0.2047       169
           2     0.3893    0.4090    0.3989      1719
           3     0.6434    0.6179    0.6304      5914
           4     0.5387    0.6231    0.5778      3343
           5     0.3032    0.1187    0.1707       480

    accuracy                         0.5611     11625
   macro avg     0.4361    0.3845    0.3965     11625
weighted avg     0.5568    0.5611    0.5559     11625



,pred_1,pred_2,pred_3,pred_4,pred_5
true_1,26,48,55,38,2
true_2,31,703,792,186,7
true_3,19,915,3654,1289,37
true_4,4,110,1061,2083,85
true_5,5,30,117,271,57


In [6]:
# Inference on full dataset and export raw logits + class probabilities
full_texts = data["text_clean"].fillna("")
full_labels = data["target_triage_acuity"].astype(int)

full_ds = OrdinalTextDataset(full_texts, full_labels, tokenizer, max_len=MAX_LEN)
full_loader = DataLoader(full_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

model.eval()
all_logits = []
with torch.no_grad():
    for batch in tqdm(full_loader, desc="Full-data inference", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(logits.detach().cpu())

full_logits = torch.cat(all_logits, dim=0)
cond_probs = torch.sigmoid(full_logits)
cum_probs = torch.cumprod(cond_probs, dim=1)

# Convert cumulative P(y > k) to class probabilities P(y = c).
class_probs = torch.zeros((cum_probs.size(0), NUM_CLASSES), dtype=cum_probs.dtype)
class_probs[:, 0] = 1.0 - cum_probs[:, 0]
for c in range(1, NUM_CLASSES - 1):
    class_probs[:, c] = cum_probs[:, c - 1] - cum_probs[:, c]
class_probs[:, NUM_CLASSES - 1] = cum_probs[:, NUM_CLASSES - 2]
class_probs = torch.clamp(class_probs, min=0.0)
class_probs = class_probs / class_probs.sum(dim=1, keepdim=True).clamp(min=1e-12)

pred_class = class_probs.argmax(dim=1).numpy() + 1

out_df = pd.DataFrame({
    "row_index": data.index.to_numpy(),
})

for k in range(NUM_CLASSES - 1):
    out_df[f"raw_logit_t{k+1}"] = full_logits[:, k].numpy()
for c in range(NUM_CLASSES):
    out_df[f"prob_class_{c+1}"] = class_probs[:, c].numpy()

output_path = PROJECT_ROOT / "working_data" / "nlp_full_logits_probs.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
out_df.to_csv(output_path, index=False)

print(f"Saved {len(out_df):,} rows to: {output_path}")
display(out_df.head())

Full-data inference:   0%|          | 0/3633 [00:00<?, ?it/s]

Saved 58,124 rows to: /home/gaurav/python/kaggle/triage/working_data/nlp_full_logits_probs.csv


,row_index,raw_logit_t1,raw_logit_t2,raw_logit_t3,raw_logit_t4,prob_class_1,prob_class_2,prob_class_3,prob_class_4,prob_class_5
0,0,4.101734,3.252468,1.347074,-1.566475,0.016275,0.036632,0.195432,0.621835,0.129827
1,1,4.115688,2.771565,1.061144,-2.035211,0.016053,0.057935,0.238069,0.608447,0.079495
2,2,4.264779,3.028163,1.219005,-1.332983,0.013860,0.045530,0.214564,0.574545,0.151501
3,3,2.998273,1.222919,-2.706558,-0.923420,0.047504,0.216620,0.689820,0.032965,0.013092
4,4,4.033425,2.606474,1.206017,-1.958915,0.017405,0.067527,0.210838,0.617198,0.087032
